# AT1 - Perceptron e KNN em Pratica

## Desafio 1


In [ ]:
import numpy as np


def treinar_perceptron(X, y, taxa_aprendizado, epocas):
    """Treina o Perceptron usando a regra de ajuste de Rosenblatt."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    if X.ndim != 2 or y.ndim != 1 or X.shape[0] != y.shape[0]:
        raise ValueError("X deve ser uma matriz e y deve ter um r?tulo por amostra.")
    if not np.isin(y, [0, 1]).all():
        raise ValueError("Os r?tulos devem ser 0 ou 1.")
    if taxa_aprendizado <= 0 or epocas <= 0:
        raise ValueError("A taxa deve ser positiva e epocas deve ser maior que zero.")

    pesos = np.ones(X.shape[1], dtype=float)
    vies = 1.0
    for _ in range(epocas):
        quantidade_erros = 0
        for amostra, classe_real in zip(X, y):
            combinacao_linear = np.dot(amostra, pesos) + vies
            classe_prevista = int(combinacao_linear >= 0)
            erro = classe_real - classe_prevista
            if erro != 0:
                pesos += taxa_aprendizado * erro * amostra
                vies += taxa_aprendizado * erro
                quantidade_erros += 1
        if quantidade_erros == 0:
            break
    return pesos, vies


def predizer_perceptron(amostra, pesos, vies):
    """Aplica a combina??o linear e a fun??o degrau."""
    amostra = np.asarray(amostra, dtype=float)
    pesos = np.asarray(pesos, dtype=float)
    if amostra.ndim != 1 or amostra.size != pesos.size:
        raise ValueError("A amostra deve ter a mesma quantidade de atributos dos pesos.")
    return int(np.dot(amostra, pesos) + vies >= 0)


X_treino_fraude = np.array([
    [1.5, 1.0], [2.0, 2.0], [3.5, 1.5], [3.0, 3.0],
    [6.5, 5.0], [7.0, 7.0], [8.5, 6.0], [9.0, 8.0]
])
y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])
transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

pesos_fraude, vies_fraude = treinar_perceptron(
    X_treino_fraude, y_treino_fraude, taxa_aprendizado=0.01, epocas=1000
)
predicao_A = predizer_perceptron(transacao_A, pesos_fraude, vies_fraude)
predicao_B = predizer_perceptron(transacao_B, pesos_fraude, vies_fraude)
rotulos_fraude = {0: "Leg?tima", 1: "Suspeita (Risco de Fraude)"}

print("=== DESAFIO 1: PERCEPTRON ===")
print(f"Pesos finais: {np.round(pesos_fraude, 4)}")
print(f"Vi?s final: {vies_fraude:.4f}")
print(f"Transa??o A: classe {predicao_A} - {rotulos_fraude[predicao_A]}")
print(f"Transa??o B: classe {predicao_B} - {rotulos_fraude[predicao_B]}")

=== DESAFIO 1: PERCEPTRON ===
Pesos finais: [-0.005  0.055]
Vi?s final: -0.1900
Transa??o A: classe 1 - Suspeita (Risco de Fraude)
Transa??o B: classe 1 - Suspeita (Risco de Fraude)


## Desafio 2 - 


In [2]:
def classificar_knn(X_treino, y_treino, ponto, k, metrica):
    """Retorna classe prevista, ?ndices dos vizinhos e suas dist?ncias."""
    X_treino = np.asarray(X_treino, dtype=float)
    y_treino = np.asarray(y_treino, dtype=int)
    ponto = np.asarray(ponto, dtype=float)
    metrica = metrica.lower()
    if X_treino.ndim != 2 or ponto.ndim != 1 or X_treino.shape[1] != ponto.size:
        raise ValueError("As dimens?es do treino e do ponto n?o s?o compat?veis.")
    if y_treino.size != X_treino.shape[0]:
        raise ValueError("Deve existir um r?tulo para cada amostra.")
    if not 1 <= k <= X_treino.shape[0]:
        raise ValueError("k deve estar entre 1 e o n?mero de amostras.")

    diferencas = X_treino - ponto
    if metrica == "euclidiana":
        distancias = np.sqrt(np.sum(diferencas ** 2, axis=1))
    elif metrica == "manhattan":
        distancias = np.sum(np.abs(diferencas), axis=1)
    else:
        raise ValueError("A m?trica deve ser 'euclidiana' ou 'manhattan'.")

    indices_vizinhos = np.argsort(distancias, kind="stable")[:k]
    classes, votos = np.unique(y_treino[indices_vizinhos], return_counts=True)
    classe_predita = int(classes[np.argmax(votos)])
    return classe_predita, indices_vizinhos, distancias[indices_vizinhos]


X_treino_churn = np.array([
    [2.0, 1.0], [3.0, 0.0], [5.0, 1.0], [6.0, 2.0],
    [15.0, 4.0], [18.0, 5.0], [20.0, 4.0], [22.0, 6.0]
])
y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])
cliente_1 = np.array([13.0, 10.0])
cliente_2 = np.array([7.0, 1.5])
rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}

resultado_cliente_1 = classificar_knn(
    X_treino_churn, y_treino_churn, cliente_1, k=3, metrica="euclidiana"
)
resultado_cliente_2 = classificar_knn(
    X_treino_churn, y_treino_churn, cliente_2, k=5, metrica="manhattan"
)

print("=== DESAFIO 2: KNN ===")
for nome, ponto, resultado, k, metrica in [
    ("Cliente 1", cliente_1, resultado_cliente_1, 3, "Euclidiana"),
    ("Cliente 2", cliente_2, resultado_cliente_2, 5, "Manhattan"),
]:
    classe, indices, distancias = resultado
    print(f"\n{nome}: {ponto.tolist()} | k={k} | m?trica={metrica}")
    print(f"Classe prevista: {classe} - {rotulos_churn[classe]}")
    print(f"?ndices dos vizinhos: {indices}")
    print(f"Dist?ncias: {np.round(distancias, 4)}")

=== DESAFIO 2: KNN ===

Cliente 1: [13.0, 10.0] | k=3 | m?trica=Euclidiana
Classe prevista: 1 - Alto Risco
?ndices dos vizinhos: [4 5 6]
Dist?ncias: [6.3246 7.0711 9.2195]

Cliente 2: [7.0, 1.5] | k=5 | m?trica=Manhattan
Classe prevista: 0 - Baixo Risco
?ndices dos vizinhos: [3 2 0 1 4]
Dist?ncias: [ 1.5  2.5  5.5  5.5 10.5]


## Desafio 3 

In [4]:
def recomendar_servidores(catalogo, demanda, quantidade):
    """Retorna os ?ndices e as menores dist?ncias Euclidianas."""
    catalogo = np.asarray(catalogo, dtype=float)
    demanda = np.asarray(demanda, dtype=float)
    if catalogo.ndim != 2 or demanda.ndim != 1 or catalogo.shape[1] != demanda.size:
        raise ValueError("Cat?logo e demanda devem ter tr?s atributos compat?veis.")
    if not 1 <= quantidade <= catalogo.shape[0]:
        raise ValueError("A quantidade deve estar entre 1 e o tamanho do cat?logo.")

    distancias = np.linalg.norm(catalogo - demanda, axis=1)
    indices = np.argsort(distancias, kind="stable")[:quantidade]
    return indices, distancias[indices]


catalogo_servidores = np.array([
    [2.0, 4.0, 50.0], [4.0, 8.0, 100.0], [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0], [32.0, 64.0, 1000.0], [64.0, 128.0, 2000.0]
])
nomes_servidores = [
    "Micro Inst?ncia Web", "Standard App Server", "Medium Backend & Cache",
    "Database Enterprise", "High Performance Computing", "Big Data & AI Training"
]
perfil_hardware = np.array([12.0, 28.0, 850.0])
quantidade_recomendacoes = 2
indices_recomendados, distancias_recomendadas = recomendar_servidores(
    catalogo_servidores, perfil_hardware, quantidade_recomendacoes
)

print("=== DESAFIO 3: RECOMENDA??O DE SERVIDORES ===")
print(f"Perfil de hardware: {perfil_hardware.tolist()}")
for posicao, (indice, distancia) in enumerate(
    zip(indices_recomendados, distancias_recomendadas), start=1
):
    especificacao = catalogo_servidores[indice].astype(int).tolist()
    print(f"\n{posicao}? lugar: {nomes_servidores[indice]}")
    print(f"Especifica??es [vCPUs, RAM GB, SSD GB]: {especificacao}")
    print(f"Dist?ncia Euclidiana: {distancia:.4f}")

=== DESAFIO 3: RECOMENDA??O DE SERVIDORES ===
Perfil de hardware: [12.0, 28.0, 850.0]

1? lugar: High Performance Computing
Especifica??es [vCPUs, RAM GB, SSD GB]: [32, 64, 1000]
Dist?ncia Euclidiana: 155.5506

2? lugar: Database Enterprise
Especifica??es [vCPUs, RAM GB, SSD GB]: [16, 32, 500]
Dist?ncia Euclidiana: 350.0457
